# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a practical guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
print("Dataset loaded!")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

Let's enumerate all record sets, show their `@id`s, names/titles, and associated field details. This helps guide what data is available for extraction.


In [ ]:
# Get all record sets with their @ids and fields
record_sets = dataset.metadata.record_sets
print(f"Number of record sets found: {len(record_sets)}\n")
for record_set in record_sets:
    print(f"Record Set @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', '')}")
    print(f"  Description: {record_set.get('description', '')}")
    if 'field' in record_set:
        print("  Fields:")
        for field in record_set['field']:
            if isinstance(field, dict):
                print(f"    Field @id: {field.get('@id', '?')}")
                print(f"      Name: {field.get('name', '')}")
                print(f"      Data type: {field.get('dataType', '')}")
            else:
                print(f"    Field Reference @id: {field}")
    else:
        print("  No fields found on this record set.")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# List all available record set @ids
rs_ids = [r['@id'] for r in record_sets]
print(f"Record set @ids found: {rs_ids}")

# For demonstration, let's extract data from all record sets
dataframes = {}
for rs_id in rs_ids:
    try:
        df = pd.DataFrame(dataset.records(record_set=rs_id))
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records from {rs_id}")
    except Exception as e:
        print(f"Could not load records from {rs_id}: {e}")

# If dataframes are loaded, print columns for the first major record set
if dataframes:
    first_rs_id = next(iter(dataframes))
    print(f"\nColumns in record set {first_rs_id}: ")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps like filtering, normalization, and grouping to a numeric field.

**Note:** You need to know the correct field `@id` and column names; adjust according to your exploration.


In [ ]:
# Example EDA
# Choose a record set (by @id) that holds tabular data with numeric fields for demonstration

import numpy as np

# Pick the first record set loaded that has at least one float/integer column
selected_rs_id = None
numeric_field = None
for rs_id, df in dataframes.items():
    numeric_candidates = df.select_dtypes(include=[np.number]).columns
    if len(numeric_candidates) > 0:
        selected_rs_id = rs_id
        numeric_field = numeric_candidates[0]
        break

if selected_rs_id is not None and numeric_field is not None:
    print(f"Using record set: {selected_rs_id}, numeric field: {numeric_field}")

    # Filtering records above the mean
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}):")
    display(filtered_df.head())

    # Normalizing the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a text/categorical column if available
    group_field = None
    for col in df.select_dtypes(include=[object]).columns:
        if col != numeric_field:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (mean values):")
        display(grouped_df.head())
else:
    print("No suitable record set with numeric columns found for EDA.")

## 5. Visualization
Visualize distributions (e.g., histogram, boxplot) of numeric fields, or relationships between numeric and categorical fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_rs_id and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} grouped by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, you explored the structure of the FAIR² mass spectrometry dataset using Croissant metadata, loaded the data dynamically, and performed simple exploratory and statistical analysis on chosen fields. For additional information, refer to the dataset documentation, and tailor the data extraction and analysis to your use case.
